# UVA DS4002 - MET Art Image Classification

### Image scraping code that attempts to download images classified as paintings from the MET website api

In [ ]:
# Read in the dataset from the project guthub repository
import pandas as pd

url = 'https://raw.githubusercontent.com/dsizzle67/Project3-DS4002/main/DATA/MetObjects_FINAL.csv'

df_paintings = pd.read_csv(url)

df_paintings.head()

,Object Number,Title,Period,Classification,Link Resource
0,12.37.135,NaN,Qing dynasty (1644–1911),Paintings,http://www.metmuseum.org/art/collection/search...
1,13.100.22,明 丁雲鵬 潯陽送客圖 軸|Song of the Lute,late Ming dynasty (1368–1644),Paintings,http://www.metmuseum.org/art/collection/search...
2,13.100.25,清 佚名 倣王翬 倣李成山水圖 軸|Landscape after Li C...,Qing dynasty (1644–1911),Paintings,http://www.metmuseum.org/art/collection/search...
3,13.100.40,清 佚名 倣王翬 倣仇實父採菱圖 扇|Gathering Water Che...,Qing dynasty (1644–1911),Paintings,http://www.metmuseum.org/art/collection/search...
4,13.100.42,清 佚名 倣王鑑 倣惠崇山水圖 扇|Landscape in the Sty...,Qing dynasty (1644–1911),Paintings,http://www.metmuseum.org/art/collection/search...


In [ ]:
# Define a function to download an image from a provided URL
import os
import requests

def download_image(image_url, file_dir):
    response = requests.get(image_url)

    if response.status_code == 200:
        directory = os.path.dirname(file_dir)
        os.makedirs(directory, exist_ok=True)

        with open(file_dir, "wb") as fp:
            fp.write(response.content)
        print("Image downloaded successfully.")
    else:
        print(f"Failed to download the image. Status code: {response.status_code}")

In [ ]:
#  Loop through the dataset, extract Object ID and call MET API to and download the image
import requests
from PIL import Image
from io import BytesIO
import time

# 1. Extract the Object ID from the end of the Link Resource URL
df_paintings['Object_ID'] = df_paintings['Link Resource'].str.split('/').str[-1]

# test_df = df_paintings.head(10).copy()

data_folder = "../DATA/IMAGES/"
failed_object_ids = []

for index, row in df_paintings.iterrows():
    obj_id = row['Object_ID']
    img_folder = ""
    # Identify which period the painting is from and assign it to the correct folder
    if 'Qing' in str(df_paintings.at[index, 'Period']) or 'Ming' in str(df_paintings.at[index, 'Period']):
        img_folder = "Qing_Ming"
    elif 'Edo' in str(df_paintings.at[index, 'Period']):
        img_folder = "Edo"
    elif 'Meiji' in str(df_paintings.at[index, 'Period']):
        img_folder = "Meiji"
    else:
        img_folder = "Other_Periods"

    # Don't download images that have already been downloaded (this is mainly useful when retrying to download failed images)
    dest_path = f"{data_folder}{img_folder}/{obj_id}.jpg"
    if os.path.exists(dest_path):
        print(f"Already downloaded: {obj_id}.jpg -> {img_folder}")
        continue

    # The Met's public API endpoint for getting object data
    api_url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{obj_id}"
    
    try:
        # Call API to get the JSON data for this specific painting
        response = requests.get(api_url)
        if response.status_code != 200:
            print(f"API request failed with status code {response.status_code} for Object ID {obj_id}")
            failed_object_ids.append(obj_id)
            continue
        response = response.json()
        
        # Look for the primary image URL
        img_url = response.get('primaryImageSmall') or response.get('primaryImage')

        if not img_url:
            print(f"No image URL found for Object ID {obj_id}")
            failed_object_ids.append(obj_id)
            continue

        download_image(img_url, dest_path)
        print(f"Image {obj_id}.jpg downloaded to folder ", img_folder)
            
        # Pause briefly to avoid overloading the MET's servers
        time.sleep(0.1) 
        
    except Exception as e:
        print(f"Failed to process Object ID {obj_id}: {e}")
        failed_object_ids.append(obj_id)

print("\nFailed object IDs:", failed_object_ids)
print(f"Total failed: {len(failed_object_ids)}")

Already downloaded: 35969.jpg -> Qing_Ming
Already downloaded: 35970.jpg -> Qing_Ming
Already downloaded: 35971.jpg -> Qing_Ming
Already downloaded: 35972.jpg -> Qing_Ming
Already downloaded: 35973.jpg -> Qing_Ming
Already downloaded: 35974.jpg -> Qing_Ming
Already downloaded: 35975.jpg -> Qing_Ming
Already downloaded: 35977.jpg -> Qing_Ming
Already downloaded: 35979.jpg -> Qing_Ming
Already downloaded: 35981.jpg -> Qing_Ming
Already downloaded: 35984.jpg -> Qing_Ming
Already downloaded: 35986.jpg -> Qing_Ming
Already downloaded: 35988.jpg -> Qing_Ming
Already downloaded: 35989.jpg -> Qing_Ming
Already downloaded: 35990.jpg -> Qing_Ming
Already downloaded: 35992.jpg -> Qing_Ming
Already downloaded: 35994.jpg -> Qing_Ming
Already downloaded: 35996.jpg -> Qing_Ming
Already downloaded: 35998.jpg -> Qing_Ming
Already downloaded: 36000.jpg -> Qing_Ming
Already downloaded: 36001.jpg -> Qing_Ming
Already downloaded: 36002.jpg -> Qing_Ming
Already downloaded: 36004.jpg -> Qing_Ming
Already dow

In [13]:
# Retry failed downloads by reusing the failed_object_ids list from the main download loop.
failed_object_ids = globals().get("failed_object_ids", [])

if not failed_object_ids:
    print("No failed_object_ids found. Please run the main download loop first.")
else:
    print(f"Retrying {len(failed_object_ids)} failed object ids")


def retry_failed_images(object_ids, df, data_folder="../DATA/IMAGES/"):
    session = requests.Session()
    session.headers.update({"User-Agent": "DS4002 Retry Script"})
    retry_failures = []

    for obj_id in object_ids:
        row = df[df["Object_ID"] == obj_id]
        if row.empty:
            print(f"Object ID {obj_id} not found in dataframe")
            retry_failures.append(obj_id)
            continue

        period = str(row.iloc[0]["Period"])
        api_url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{obj_id}"

        try:
            response = session.get(api_url, timeout=20)
            if response.status_code != 200:
                print(f"Retry API request failed with status {response.status_code} for {obj_id}")
                retry_failures.append(obj_id)
                continue

            data = response.json()
            img_url = data.get("primaryImageSmall") or data.get("primaryImage")
            if not img_url:
                print(f"No image URL found on retry for {obj_id}")
                retry_failures.append(obj_id)
                continue

            if "Qing" in period or "Ming" in period:
                img_folder = "Qing_Ming"
            elif "Edo" in period:
                img_folder = "Edo"
            elif "Meiji" in period:
                img_folder = "Meiji"
            else:
                img_folder = "Other_Periods"

            download_image(img_url, f"{data_folder}{img_folder}/{obj_id}.jpg")
            print(f"Retry download succeeded for {obj_id} -> {img_folder}")
            time.sleep(0.1)

        except Exception as e:
            print(f"Retry failed for {obj_id}: {e}")
            retry_failures.append(obj_id)

    return retry_failures

if failed_object_ids:
    retry_failures = retry_failed_images(failed_object_ids, df_paintings, data_folder="../DATA/IMAGES/")
    print("\nRetry failures:", retry_failures)
    print(f"Total retry failures: {len(retry_failures)}")

Retrying 2195 failed object ids
Retry API request failed with status 403 for 36061
No image URL found on retry for 36068
No image URL found on retry for 36119
No image URL found on retry for 36169
No image URL found on retry for 36182
No image URL found on retry for 36183
No image URL found on retry for 36185
No image URL found on retry for 36189
No image URL found on retry for 36190
No image URL found on retry for 36194
No image URL found on retry for 36196
No image URL found on retry for 36197
No image URL found on retry for 36198
No image URL found on retry for 36199
No image URL found on retry for 36203
No image URL found on retry for 36209
No image URL found on retry for 36216
No image URL found on retry for 36261
No image URL found on retry for 36277
No image URL found on retry for 36321
Image downloaded successfully.
Retry download succeeded for 36457 -> Other_Periods
Image downloaded successfully.
Retry download succeeded for 36459 -> Other_Periods
Image downloaded successfully

In [ ]:
# Retry failed downloads again
failed_object_ids = retry_failures

if failed_object_ids:
    retry_failures = retry_failed_images(failed_object_ids, df_paintings, data_folder="../DATA/IMAGES/")
    print("\nRetry failures:", retry_failures)
    print(f"Total retry failures: {len(retry_failures)}")

No image URL found on retry for 36061
No image URL found on retry for 36068
No image URL found on retry for 36119
No image URL found on retry for 36169
No image URL found on retry for 36182
No image URL found on retry for 36183
No image URL found on retry for 36185
No image URL found on retry for 36189
No image URL found on retry for 36190
No image URL found on retry for 36194
No image URL found on retry for 36196
No image URL found on retry for 36197
No image URL found on retry for 36198
No image URL found on retry for 36199
No image URL found on retry for 36203
No image URL found on retry for 36209
No image URL found on retry for 36216
No image URL found on retry for 36261
No image URL found on retry for 36277
No image URL found on retry for 36321
No image URL found on retry for 38026
Retry API request failed with status 404 for 39714
Image downloaded successfully.
Retry download succeeded for 39715 -> Qing_Ming
Image downloaded successfully.
Retry download succeeded for 39717 -> Qin

In [ ]:
# Retry failed downloads again
failed_object_ids = retry_failures

if failed_object_ids:
    retry_failures = retry_failed_images(failed_object_ids, df_paintings, data_folder="../DATA/IMAGES/")
    print("\nRetry failures:", retry_failures)
    print(f"Total retry failures: {len(retry_failures)}")

No image URL found on retry for 36061
No image URL found on retry for 36068
No image URL found on retry for 36119
No image URL found on retry for 36169
No image URL found on retry for 36182
No image URL found on retry for 36183
No image URL found on retry for 36185
No image URL found on retry for 36189
No image URL found on retry for 36190
No image URL found on retry for 36194
No image URL found on retry for 36196
No image URL found on retry for 36197
No image URL found on retry for 36198
No image URL found on retry for 36199
No image URL found on retry for 36203
No image URL found on retry for 36209
No image URL found on retry for 36216
No image URL found on retry for 36261
No image URL found on retry for 36277
No image URL found on retry for 36321
No image URL found on retry for 38026
Retry API request failed with status 404 for 39714
Image downloaded successfully.
Retry download succeeded for 40087 -> Other_Periods
Image downloaded successfully.
Retry download succeeded for 40088 ->